<a href="https://colab.research.google.com/github/Sergiodzm99/Proyect-Model_MLB/blob/main/DF_EDA_API_Stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='00FF00'> API Stats

In [ ]:
import requests
import pandas as pd

In [ ]:
#2023
url = (
    "https://statsapi.mlb.com/api/v1/schedule"
    "?sportId=1"
    "&startDate=2023-01-01"
    "&endDate=2023-12-31"
)
response = requests.get(url)
data = response.json()

# Extraer juegos
games = [game for day in data["dates"] for game in day["games"]]

df_2023 = pd.json_normalize(games)

In [ ]:
#2024
url = (
    "https://statsapi.mlb.com/api/v1/schedule"
    "?sportId=1"
    "&startDate=2024-01-01"
    "&endDate=2024-12-31"
)
response = requests.get(url)
data = response.json()

# Extraer juegos
games = [game for day in data["dates"] for game in day["games"]]

df_2024 = pd.json_normalize(games)

In [ ]:
#2025
url = (
    "https://statsapi.mlb.com/api/v1/schedule"
    "?sportId=1"
    "&startDate=2025-01-01"
    "&endDate=2025-12-31"
)
response = requests.get(url)
data = response.json()

# Extraer juegos
games = [game for day in data["dates"] for game in day["games"]]

df_2025 = pd.json_normalize(games)

In [ ]:
#2026
url = (
    "https://statsapi.mlb.com/api/v1/schedule"
    "?sportId=1"
    "&startDate=2026-01-01"
    "&endDate=2026-05-31"
)
response = requests.get(url)
data = response.json()

# Extraer juegos
games = [game for day in data["dates"] for game in day["games"]]

df_2026 = pd.json_normalize(games)

In [ ]:
df = pd.concat([df_2023, df_2024, df_2025, df_2026], ignore_index=True)

In [ ]:
df['home_win'] = (df['teams.home.score'] > df['teams.away.score']).astype(int)

# 1 = ganó el equipo local
# 0 = ganó el equipo visitante

# <font color='00FF00'> Leakage Temporal
</font>

### <font color='00FFFF'> Fun para Comprobar leakage temporal

In [ ]:
def check_leakage_simple(df):
    base_cols = ['officialDate', 'gameDate']

    team_sides = {
        'home': {
            'team': 'teams.home.team.name',
            'wins': 'teams.home.leagueRecord.wins',
            'losses': 'teams.home.leagueRecord.losses',
        },
        'away': {
            'team': 'teams.away.team.name',
            'wins': 'teams.away.leagueRecord.wins',
            'losses': 'teams.away.leagueRecord.losses',
        },
    }

    games_by_side = []

    for side, cols in team_sides.items():
        side_games = (
            df[base_cols + list(cols.values())]
            .rename(columns={v: k for k, v in cols.items()})
            .assign(side=side)
        )

        games_by_side.append(side_games)

    games = pd.concat(games_by_side, ignore_index=True)

    first_games = (
        games
        .sort_values(['team', 'officialDate', 'gameDate'])
        .groupby('team', as_index=False)
        .first()
    )

    leakage = first_games[
        (first_games['wins'] != 0) |
        (first_games['losses'] != 0)
    ]

    if leakage.empty:
        print("✅ No se detectó leakage temporal.")
    else:
        print("🚨 Posible leakage temporal detectado.")
        print(f"Equipos sospechosos: {len(leakage)}")

    return leakage

### <font color='00FFFF'> Comprobar leakage temporal

In [ ]:
leakage = check_leakage_simple(df)
leakage.tail()

🚨 Posible leakage temporal detectado.
Equipos sospechosos: 64


,team,officialDate,gameDate,wins,losses,side
60,United States,2023-03-08,2023-03-09T02:05:00Z,0,1,away
61,Venezuela,2023-03-08,2023-03-08T23:05:00Z,1,0,away
62,Washington Nationals,2023-02-25,2023-02-25T18:05:00Z,1,0,away
63,Washington Nationals Prospects,2024-03-26,2024-03-26T16:05:00Z,0,2,away
64,West Virginia Mountaineers,2023-02-27,2023-02-28T01:40:00Z,0,1,away


### <font color='00FFFF'> Construcción de Variables sin leakage

In [ ]:
#Ordenar cronologicamente

df = df.sort_values(
    ['season','officialDate', 'gameDate']
).reset_index(drop=True)

In [ ]:
#Listas para almacenar valores pre-partido

home_wins_before = []
home_losses_before = []

away_wins_before = []
away_losses_before = []

In [ ]:
# Reconstrucción de récord por temporada

from collections import defaultdict

current_season = None

for _, row in df.iterrows():

    # ----------------------------------
    # Reiniciar al cambiar temporada
    # ----------------------------------

    if row['season'] != current_season:

        current_season = row['season']

        team_records = defaultdict(
            lambda: {
                'wins': 0,
                'losses': 0
            }
        )

    home_team = row['teams.home.team.name']
    away_team = row['teams.away.team.name']

    # ----------------------------------
    # Guardar récord PREVIO
    # ----------------------------------

    home_wins_before.append(
        team_records[home_team]['wins']
    )

    home_losses_before.append(
        team_records[home_team]['losses']
    )

    away_wins_before.append(
        team_records[away_team]['wins']
    )

    away_losses_before.append(
        team_records[away_team]['losses']
    )

    # ----------------------------------
    # Actualizar con resultado del juego
    # ----------------------------------

    if row['home_win'] == 1:

        team_records[home_team]['wins'] += 1
        team_records[away_team]['losses'] += 1

    else:

        team_records[home_team]['losses'] += 1
        team_records[away_team]['wins'] += 1


In [ ]:
# Agregar columnas
# ==================================================

df['home_wins_before'] = home_wins_before
df['home_losses_before'] = home_losses_before

df['away_wins_before'] = away_wins_before
df['away_losses_before'] = away_losses_before

# ==================================================
# Porcentajes PRE-partido
# ==================================================

home_games = (df['home_wins_before'] + df['home_losses_before'])

away_games = (df['away_wins_before'] + df['away_losses_before'])

df['home_pct_before'] = (df['home_wins_before'] / home_games.replace(0, 1))

df['away_pct_before'] = (df['away_wins_before'] / away_games.replace(0, 1))

In [ ]:
df.groupby('season')[[
    'home_wins_before',
    'home_losses_before',
    'away_wins_before',
    'away_losses_before'
]].first()

,home_wins_before,home_losses_before,away_wins_before,away_losses_before
season,,,,
2023,0,0,0,0
2024,0,0,0,0
2025,0,0,0,0
2026,0,0,0,0


In [ ]:
drop_cols = [

    'teams.home.leagueRecord.wins',
    'teams.home.leagueRecord.losses',
    'teams.home.leagueRecord.pct',

    'teams.away.leagueRecord.wins',
    'teams.away.leagueRecord.losses',
    'teams.away.leagueRecord.pct'
]

df = df.drop(columns=drop_cols)

# <font color='00FF00'> Selección de Variables
</font>

### <font color='00FFFF'> Filtrado de variables no informativas


In [ ]:
variables_numericas = df.select_dtypes(include=['number'])
variables_categoricas = df.select_dtypes(include=['object', 'category', 'bool'])

In [ ]:
def eliminar_categoricas_inutiles(df):
    """
    Eliminar columnas categóricas que:
    1. Tienen un solo valor único
    2. Tienen tantos valores únicos como filas del DataFrame
    """
    columnas_eliminar = []
    n_filas = len(df)

    for col in variables_categoricas:
        n_unicos = df[col].nunique(dropna=False)

        if n_unicos == 1 or n_unicos == n_filas:
            columnas_eliminar.append(col)

    print("Columnas eliminadas:", columnas_eliminar)

    return df.drop(columns=columnas_eliminar)

In [ ]:
df = eliminar_categoricas_inutiles(df)

Columnas eliminadas: ['publicFacing', 'tiebreaker', 'calendarEventID', 'recordSource', 'ifNecessary', 'ifNecessaryDescription', 'status.abstractGameState', 'status.abstractGameCode']


### <font color='00FFFF'> Selección de variables por dominio y calidad

In [ ]:
features = [

    # Contexto del juego
    'gameType',
    'dayNight',
    'doubleHeader',
    'gameNumber',
    'scheduledInnings',
    'gamesInSeries',
    'seriesGameNumber',

    # Equipo visitante
    'teams.away.team.name',
    'away_wins_before',
    'away_losses_before',
    'away_pct_before',

    # Equipo local
    'teams.home.team.name',
    'home_wins_before',
    'home_losses_before',
    'home_pct_before',

    # Estadio
    'venue.name',

    # Target
    'home_win',

    #Primary Key - ID
    'gamePk'
]

In [ ]:
df = df[features].copy()

### <font color='00FFFF'> Modificar formato de variables

# <font color='00FF00'> Creación de Nuevas Variables API Stats
</font>

In [ ]:
df['win_pct_diff'] = (df['home_pct_before'] - df['away_pct_before'])

df['wins_diff'] = (df['home_wins_before'] - df['away_wins_before'])

df['losses_diff'] = (df['home_losses_before'] - df['away_losses_before'])

df['home_games_played'] = (df['home_wins_before'] + df['home_losses_before'])

df['away_games_played'] = (df['away_wins_before'] + df['away_losses_before'])

# <font color='00FF00'> EDA
</font>

###  Nulos

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)

,0
gamesInSeries,64
seriesGameNumber,64


### <font color='00FFFF'> Describe

In [ ]:
df.describe(include='number')

,gameNumber,scheduledInnings,gamesInSeries,seriesGameNumber,away_wins_before,away_losses_before,away_pct_before,home_wins_before,home_losses_before,home_pct_before,home_win,gamePk,win_pct_diff,wins_diff,losses_diff,home_games_played,away_games_played
count,10391.000000,10391.000000,10327.000000,10327.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000,10391.000000
mean,1.010105,8.998941,2.813692,1.897356,46.114330,45.562795,0.496978,46.043018,45.696276,0.498250,0.518333,758296.427485,0.001272,-0.071312,0.133481,91.739294,91.677124
std,0.100019,0.049041,0.973876,0.933857,30.138404,29.594033,0.114953,30.094263,29.624141,0.111774,0.499688,35469.114539,0.157079,10.640427,11.086960,57.769114,57.797869
min,1.000000,7.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,716352.000000,-1.000000,-56.000000,-65.000000,0.000000,0.000000
25%,1.000000,9.000000,3.000000,1.000000,21.000000,20.000000,0.451220,21.000000,20.000000,0.450820,0.000000,719046.500000,-0.071795,-5.000000,-5.000000,42.000000,42.000000
50%,1.000000,9.000000,3.000000,2.000000,42.000000,42.000000,0.500000,42.000000,42.000000,0.500000,1.000000,746991.000000,0.000000,0.000000,0.000000,85.000000,85.000000
75%,1.000000,9.000000,3.000000,3.000000,70.000000,69.000000,0.550562,70.000000,70.000000,0.550725,1.000000,777900.500000,0.074367,5.000000,5.000000,141.000000,141.000000
max,2.000000,10.000000,7.000000,7.000000,129.000000,146.000000,1.000000,128.000000,145.000000,1.000000,1.000000,836150.000000,1.000000,62.000000,58.000000,216.000000,213.000000


In [ ]:
df.describe(include='object')

,gameType,dayNight,doubleHeader,teams.away.team.name,teams.home.team.name,venue.name
count,10391,10391,10391,10391,10391,10391
unique,8,2,3,57,46,76
top,R,night,N,New York Yankees,Philadelphia Phillies,Citizens Bank Park
freq,8309,5537,10183,359,358,293


### <font color='00FFFF'> Carnalidad

In [ ]:
df.nunique().sort_values()

,0
dayNight,2
gameNumber,2
home_win,2
doubleHeader,3
scheduledInnings,3
seriesGameNumber,7
gamesInSeries,8
gameType,8
teams.home.team.name,46
teams.away.team.name,57


### <font color='00FFFF'> Distribución del Target

In [ ]:
df['home_win'].value_counts(normalize=True)

,proportion
home_win,
1,0.518333
0,0.481667


### <font color='00FFFF'> Correlaciones

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
corr = df.corr(numeric_only=True)

In [ ]:
corr['home_win'].sort_values()

,home_win
losses_diff,-0.110022
away_pct_before,-0.057029
scheduledInnings,-0.024737
away_wins_before,-0.023773
home_losses_before,-0.022387
gameNumber,-0.016225
home_games_played,-0.004431
away_games_played,-0.002766
gamesInSeries,0.003930
home_wins_before,0.013532


In [ ]:
corr_home = corr.loc[
    (corr['home_win'].abs() >= 0.05) &
    (corr.index != 'home_win'),
    'home_win'
].sort_values(key=abs, ascending=False)

corr_home

,home_win
losses_diff,-0.110022
wins_diff,0.105609
win_pct_diff,0.074007
away_pct_before,-0.057029


### <font color='00FFFF'> Multicolinealidad

Numericas:

In [ ]:
variables_corr = corr_home.index.tolist()

In [ ]:
corr_vars = df[variables_corr].corr()
corr_vars

,losses_diff,wins_diff,win_pct_diff,away_pct_before
losses_diff,1.000000,-0.949521,-0.583578,0.393264
wins_diff,-0.949521,1.000000,0.618479,-0.425468
win_pct_diff,-0.583578,0.618479,1.000000,-0.703190
away_pct_before,0.393264,-0.425468,-0.703190,1.000000


In [ ]:
#Se elimina losses_diff por su alta correlacion con wins_diff

df.drop(columns=['losses_diff'],inplace=True)

Categoricas:

In [ ]:
from scipy.stats import contingency

In [ ]:
def cramers_v(x, y):
    return contingency.association(
        pd.crosstab(x, y),
        method="cramer"
    )

In [ ]:
categorical_features = [
    'gameType',
    'dayNight',
    'doubleHeader',
    'teams.home.team.name',
    'teams.away.team.name',
    'venue.name'
]

cramer_matrix = pd.DataFrame(
    index=categorical_features,
    columns=categorical_features,
    dtype=float
)

for c1 in categorical_features:
    for c2 in categorical_features:
        cramer_matrix.loc[c1, c2] = cramers_v(
            df[c1],
            df[c2]
        )

cramer_matrix

,gameType,dayNight,doubleHeader,teams.home.team.name,teams.away.team.name,venue.name
gameType,1.000000,0.360428,0.048202,0.418351,0.511774,0.402845
dayNight,0.360428,1.000000,0.050114,0.133464,0.095846,0.398006
doubleHeader,0.048202,0.050114,1.000000,0.126955,0.082927,0.152457
teams.home.team.name,0.418351,0.133464,0.126955,1.000000,0.238270,0.876158
teams.away.team.name,0.511774,0.095846,0.082927,0.238270,1.000000,0.219408
venue.name,0.402845,0.398006,0.152457,0.876158,0.219408,1.000000


In [ ]:
cramer_target = pd.Series(
    {
        col: cramers_v(df[col], df['home_win'])
        for col in categorical_features
    }
).sort_values(ascending=False)

cramer_target

,0
venue.name,0.136755
teams.away.team.name,0.128191
teams.home.team.name,0.121450
gameType,0.025145
doubleHeader,0.017057
dayNight,0.012744


In [ ]:
#Se elimina la variable venue.name por su multicolinealidad con teams.home.team.name, las demás requieren análisis
#Esto por ser una representación más directa e interpretable del equipo local, además de ofrecer una mejor capacidad de generalización ante posibles cambios de estadio o condiciones asociadas al recinto.

categorical_features = [
    'gameType',
    'dayNight',
    'doubleHeader',
    'teams.home.team.name',
    'teams.away.team.name',
]

### <font color='00FFFF'> Outliers

Numericas:

In [ ]:
numeric_features = [

    # ==================================================
    # CONTEXTO DEL PARTIDO
    # ==================================================

    'gameNumber',
    'scheduledInnings',
    'gamesInSeries',
    'seriesGameNumber',

    # ==================================================
    # RENDIMIENTO PRE-PARTIDO
    # ==================================================

    'home_wins_before',
    'home_losses_before',
    'home_pct_before',

    'away_wins_before',
    'away_losses_before',
    'away_pct_before',

    # ==================================================
    # VARIABLES DERIVADAS
    # ==================================================

    'win_pct_diff',
    'wins_diff',

    'home_games_played',
    'away_games_played'
]

In [ ]:
outliers_iqr = {}

for col in numeric_features:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    outliers_iqr[col] = len(outliers)

outliers_iqr
#outliers['win_pct_diff'].head(2)
#outliers['win_pct_diff'].describe()
#outliers['win_pct_diff'].hist(bins=30)

{'gameNumber': 105,
 'scheduledInnings': 7,
 'gamesInSeries': 3796,
 'seriesGameNumber': 4,
 'home_wins_before': 0,
 'home_losses_before': 0,
 'home_pct_before': 611,
 'away_wins_before': 0,
 'away_losses_before': 5,
 'away_pct_before': 650,
 'win_pct_diff': 490,
 'wins_diff': 685,
 'home_games_played': 0,
 'away_games_played': 0}

In [ ]:
def filtrar_outliers(columna):
    if pd.api.types.is_numeric_dtype(columna):
        Q1 = columna.quantile(0.25)
        Q3 = columna.quantile(0.75)
        iqr = Q3 - Q1
        limite_inferior = Q1 - 1.5 * iqr
        limite_superior = Q3 + 1.5 * iqr
        return columna[(columna >= limite_inferior) & (columna <= limite_superior)]
    else:
        return columna

#datos_sin_ouliers = df[numeric_features].apply(filtrar_outliers)
#datos_sin_ouliers

Categoricas:

In [ ]:
summary = []

for col in categorical_features:

    freq = df[col].value_counts(normalize=True) * 100

    summary.append({
        'Variable': col,
        'N_Categorias': df[col].nunique(),
        'Categoria_Mayor': freq.index[0],
        'Categoria_Mayor_Pct': round(freq.iloc[0], 2)
    })

variance_cat_summary = (
    pd.DataFrame(summary)
    .sort_values('Categoria_Mayor_Pct', ascending=False)
)

variance_cat_summary

,Variable,N_Categorias,Categoria_Mayor,Categoria_Mayor_Pct
2,doubleHeader,3,N,98.00
0,gameType,8,R,79.96
1,dayNight,2,night,53.29
3,teams.home.team.name,46,Philadelphia Phillies,3.45
4,teams.away.team.name,57,New York Yankees,3.45


In [ ]:
variance_cat_summary['Variable'][variance_cat_summary['Categoria_Mayor_Pct'] >= 70]

,Variable
2,doubleHeader
0,gameType


In [ ]:
#Se elimina doubleHeader
#Se filtrara siempre usando gameType = 'R' y se elimina como variable predictora

categorical_features = [
    'dayNight',
    'teams.home.team.name',
    'teams.away.team.name',
]

In [ ]:
df = df[df['gameType'] == 'R']

df.drop(columns=['gameType','doubleHeader'],inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8309 entries, 491 to 10390
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   dayNight              8309 non-null   object 
 1   gameNumber            8309 non-null   int64  
 2   scheduledInnings      8309 non-null   int64  
 3   gamesInSeries         8309 non-null   float64
 4   seriesGameNumber      8309 non-null   float64
 5   teams.away.team.name  8309 non-null   object 
 6   away_wins_before      8309 non-null   int64  
 7   away_losses_before    8309 non-null   int64  
 8   away_pct_before       8309 non-null   float64
 9   teams.home.team.name  8309 non-null   object 
 10  home_wins_before      8309 non-null   int64  
 11  home_losses_before    8309 non-null   int64  
 12  home_pct_before       8309 non-null   float64
 13  venue.name            8309 non-null   object 
 14  home_win              8309 non-null   int64  
 15  gamePk                8

In [ ]:
df.to_parquet("df_apistats.parquet", index=False)